# Master Benchmark Suite: Full 10-Model Evaluation for Drug `M01AE`

This master notebook collects holdout predictions across all candidate models for drug `M01AE` on the 2019 Test set:
* **Part A: Point Forecast Accuracy Benchmark Table (Sorted by RMSLE)**
* **Part B: Enterprise Probabilistic Demand Range Deliverable ($[P_{10}, P_{50}, P_{90}]$)**


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'M01AE'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for M01AE loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Collect All Holdout Model Predictions & Display Benchmark Ladder
m0 = pd.read_csv('m0_naive_preds.csv')['pred_Naive'].values
m1 = pd.read_csv('m1_arima_preds.csv')['pred_ARIMA'].values
m2 = pd.read_csv('m2_ets_preds.csv')['pred_ETS'].values
m3a = pd.read_csv('m3_sarima_preds.csv')['pred_SARIMA'].values
m3b = pd.read_csv('m3_sarimax_preds.csv')['pred_SARIMAX'].values
m4 = pd.read_csv('m4_prophet_preds.csv')['pred_Prophet'].values
m5 = pd.read_csv('m5_lstm_preds.csv')['pred_LSTM'].values
m6 = pd.read_csv('m6_lightgbm_preds.csv')['pred_LightGBM'].values
m7 = pd.read_csv('m7_xgb_quantile_preds.csv')['pred_XGB_Quantile'].values
m8 = pd.read_csv('m8_tft_preds.csv')['pred_TFT'].values

m_hybrid = 0.90 * m4 + 0.10 * m6

model_dict = {
    'Hybrid Ensemble (Prophet 0.9 + LGB 0.1)': m_hybrid,
    'Model 4: Meta Prophet': m4,
    'Model 6: LightGBM + SHAP (Optuna)': m6,
    'Model 3b: SARIMAX + Exog': m3b,
    'Model 2: Holt-Winters ETS': m2,
    'Model 3a: Pure SARIMA': m3a,
    'Model 8: TFT / Deep Attention': m8,
    'Model 1: Classical ARIMA': m1,
    'Model 7: XGBoost Quantile (Optuna)': m7,
    'Model 5: PyTorch LSTM': m5,
    'Model 0: Optimised Naive (k*=365)': m0
}

records = []
for name, preds in model_dict.items():
    met = evaluate_metrics(test_series.values, preds)
    met['Model'] = name
    records.append(met)

benchmark_df = pd.DataFrame(records)[['Model', 'RMSLE', 'RMSE', 'MAE', 'WAPE (%)']].sort_values('RMSLE').reset_index(drop=True)
benchmark_df.index = benchmark_df.index + 1

print("==========================================================================")
print(f"  PART A: POINT FORECAST BENCHMARK LADDER ({TARGET_DRUG} — 2019 HOLDOUT TEST SET)")
print("==========================================================================")
display(benchmark_df)

champion = benchmark_df.iloc[0]
print("\nChampion Model for Point Forecasting:")
print(f"  * #1 Rank Model : {champion['Model']}")
print(f"  * RMSLE         : {champion['RMSLE']:.6f}")
print(f"  * RMSE          : {champion['RMSE']:.4f}")
print(f"  * MAE           : {champion['MAE']:.4f}")
print(f"  * WAPE (%)      : {champion['WAPE (%)']:.2f}%")


  PART A: POINT FORECAST BENCHMARK LADDER (M01AE — 2019 HOLDOUT TEST SET)


,Model,RMSLE,RMSE,MAE,WAPE (%)
1,Model 4: Meta Prophet,0.493317,2.315787,1.715986,44.461498
2,Hybrid Ensemble (Prophet 0.9 + LGB 0.1),0.493557,2.313437,1.712457,44.370067
3,Model 3b: SARIMAX + Exog,0.503386,2.365154,1.758796,45.570725
4,Model 6: LightGBM + SHAP (Optuna),0.505174,2.329637,1.724115,44.672114
5,Model 3a: Pure SARIMA,0.509609,2.385086,1.766353,45.766511
6,Model 8: TFT / Deep Attention,0.511197,2.291066,1.702135,44.102609
7,Model 2: Holt-Winters ETS,0.513027,2.397816,1.763614,45.695553
8,Model 1: Classical ARIMA,0.516317,2.435025,1.787235,46.307583
9,Model 7: XGBoost Quantile (Optuna),0.523633,2.289314,1.748881,45.313816
10,Model 5: PyTorch LSTM,0.526706,2.321348,1.755662,45.489512



Champion Model for Point Forecasting:
  * #1 Rank Model : Model 4: Meta Prophet
  * RMSLE         : 0.493317
  * RMSE          : 2.3158
  * MAE           : 1.7160
  * WAPE (%)      : 44.46%


In [3]:
# Step 2: Part B — Enterprise Probabilistic Demand Range Deliverable
hybrid_plan = pd.read_csv('m01ae_hybrid_supply_chain_plan.csv')

print("==========================================================================")
print(f"  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE ({TARGET_DRUG})")
print("==========================================================================")
print("First 10 Days Actionable Pack Order Ranges:")
display(hybrid_plan[['Date', 'Actual Sales', 'Lean Lower Bound (P10)', 'Expected Demand Anchor (P50)', 'Upper Target Stock (P90)', 'Order Range (Lean P10 Pack Target)', 'Order Range (Expected P50 Pack Target)', 'Order Range (Safety P90 Pack Target)']].head(10))

service_level = np.mean(test_series.values <= hybrid_plan['Upper Target Stock (P90)'].values) * 100
print(f"\nDeliverable Performance Metrics:")
print(f"  * Achieved P90 Inventory Service Level: {service_level:.2f}% (Target >= 95%)")
print(f"  * Average Daily Uncertainty Range    : {hybrid_plan['Uncertainty Band Width (P90 - P10)'].mean():.2f} units/day")


  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE (M01AE)
First 10 Days Actionable Pack Order Ranges:


,Date,Actual Sales,Lean Lower Bound (P10),Expected Demand Anchor (P50),Upper Target Stock (P90),Order Range (Lean P10 Pack Target),Order Range (Expected P50 Pack Target),Order Range (Safety P90 Pack Target)
0,2019-01-01,0.000,0.00,2.90,9.71,0,3,10
1,2019-01-02,4.397,0.87,3.11,10.81,1,4,11
2,2019-01-03,4.836,1.27,2.99,9.82,2,3,10
3,2019-01-04,3.670,1.27,3.11,9.89,2,4,10
4,2019-01-05,3.690,2.09,3.70,10.32,3,4,11
5,2019-01-06,10.010,1.72,3.69,10.42,2,4,11
6,2019-01-07,0.000,0.00,3.06,10.88,0,4,11
7,2019-01-08,3.320,1.08,3.05,12.53,2,4,13
8,2019-01-09,1.330,1.46,3.17,10.83,2,4,11
9,2019-01-10,3.580,1.41,3.05,11.48,2,4,12



Deliverable Performance Metrics:
  * Achieved P90 Inventory Service Level: 98.22% (Target >= 95%)
  * Average Daily Uncertainty Range    : 8.30 units/day
